In [1]:
import os, re, gc, json, inspect
import torch
import datasets
import transformers
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from torch.utils.tensorboard import SummaryWriter

datasets.disable_caching()
gc.collect()
torch.cuda.empty_cache()

print("transformers version:", transformers.__version__)

# Config
model_id = "facebook/wav2vec2-xls-r-300m"
lang_code = "sna"
sr = 16000
max_len = 15.0
min_len = 5.0
raw_cache_dir = "./waxal_processed_dataset"   # shared raw cache (reused by Cell 2)
vocab_path = "./wav2vec2_vocab.json"
model_out_dir = "./wav2vec2-xlsr-waxal"

NUM_MAP = {
    "1": "motsi", "2": "piri", "3": "tatu", "4": "ina", "5": "shanu",
    "6": "tanhatu", "7": "nomwe", "8": "tsere", "9": "pfumbamwe", "0": "zero"
}

def clean_text_pipeline(text):
    if text is None:
        return ""
    text = text.lower()
    for num, word in NUM_MAP.items():
        text = text.replace(num, f" {word} ")
    text = re.sub(r'[^​\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def create_raw_dataset(dataset_id, language, sample_rate, max_audio_len, min_audio_len):
    print(f"Loading {language} dataset...")
    train_pattern = f"data/ASR/{language}/{language}-train-*.parquet"
    val_pattern = f"data/ASR/{language}/{language}-validation-*.parquet"

    ds_train = datasets.load_dataset(dataset_id, name=f"{language}_asr", data_files={"train": train_pattern}, split="train")
    ds_val = datasets.load_dataset(dataset_id, name=f"{language}_asr", data_files={"validation": val_pattern}, split="validation")
    ds_combined = datasets.concatenate_datasets([ds_train, ds_val])

    print("Resampling audio...")
    ds_combined = ds_combined.cast_column("audio", datasets.Audio(sampling_rate=sample_rate))

    print("Filtering audio lengths...")
    def filter_audio_length(example):
        arr = example["audio"]["array"]
        duration = len(arr) / sample_rate
        return min_audio_len <= duration <= max_audio_len

    ds_filtered = ds_combined.filter(filter_audio_length, writer_batch_size=100)
    del ds_combined
    gc.collect()

    print("Cleaning text...")
    def clean_batch_text(batch):
        batch["transcription"] = [clean_text_pipeline(t) for t in batch["transcription"]]
        return batch

    ds_cleaned = ds_filtered.map(clean_batch_text, batched=True, batch_size=100, writer_batch_size=100)
    del ds_filtered
    gc.collect()

    print("Splitting dataset 80/20...")
    return ds_cleaned.train_test_split(test_size=0.2, seed=42)

#  Load or build raw dataset
if os.path.exists(raw_cache_dir):
    print("Loading cached raw dataset from disk...")
    raw_dataset = datasets.load_from_disk(raw_cache_dir)
else:
    print("Running raw preprocessing pipeline...")
    raw_dataset = create_raw_dataset("google/WaxalNLP", lang_code, sr, max_len, min_len)
    raw_dataset.save_to_disk(raw_cache_dir)

#  Build a character-level vocab from transcripts (XLS-R has none)
if os.path.exists(vocab_path):
    print("Loading existing vocab...")
    with open(vocab_path) as f:
        vocab_dict = json.load(f)
else:
    print("Building character vocab from training transcripts...")
    all_text = " ".join(raw_dataset["train"]["transcription"])
    vocab_list = sorted(set(all_text))
    vocab_dict = {v: i for i, v in enumerate(vocab_list)}
    vocab_dict["|"] = vocab_dict.pop(" ")   # word-boundary token, Wav2Vec2 convention
    vocab_dict["[UNK]"] = len(vocab_dict)
    vocab_dict["[PAD]"] = len(vocab_dict)
    with open(vocab_path, "w") as f:
        json.dump(vocab_dict, f)
    print(f"Vocab size: {len(vocab_dict)}")

# Build processor
tokenizer = transformers.Wav2Vec2CTCTokenizer(
    vocab_path, unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|"
)
feature_extractor = transformers.Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=sr, padding_value=0.0,
    do_normalize=True, return_attention_mask=True
)
processor = transformers.Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

#  Model
print(f"Loading {model_id}...")
model = transformers.Wav2Vec2ForCTC.from_pretrained(
    model_id,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)
model.freeze_feature_encoder()          # standard practice: CNN feature extractor stays frozen
model.gradient_checkpointing_enable()
model.config.use_cache = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

#  Lazy data collator (feature extraction happens at batch time, not upfront)
@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": feature["audio"]["array"]} for feature in features]
        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")

        label_features = [{"input_ids": self.processor.tokenizer(feature["transcription"]).input_ids} for feature in features]
        labels_batch = self.processor.pad(labels=label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

# Version-safe TrainingArguments
base_kwargs = dict(
    output_dir=model_out_dir,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    eval_strategy="steps",
    num_train_epochs=1,
    max_steps=100,
    fp16=True,
    optim="adamw_bnb_8bit",
    save_steps=50,
    eval_steps=50,
    logging_steps=10,
    learning_rate=3e-4,
    warmup_steps=50,
    save_total_limit=1,
    report_to=["tensorboard"],
    dataloader_num_workers=0,
    remove_unused_columns=False,
    # group_by_length left off: raw dataset stores audio under "audio", not
    # "input_values", so the sampler can't auto-infer lengths without an
    # extra precomputed length column.
)

sig_params = set(inspect.signature(transformers.TrainingArguments.__init__).parameters.keys())
if "eval_strategy" not in sig_params and "evaluation_strategy" in sig_params:
    base_kwargs["evaluation_strategy"] = base_kwargs.pop("eval_strategy")
final_kwargs = {k: v for k, v in base_kwargs.items() if k in sig_params}
dropped = set(base_kwargs) - set(final_kwargs)
if dropped:
    print(f"Dropped unsupported TrainingArguments kwargs: {dropped}")

training_args = transformers.TrainingArguments(**final_kwargs)
writer = SummaryWriter(log_dir='./runs/wav2vec2-xlsr')

# Version-safe Trainer
trainer_sig = set(inspect.signature(transformers.Trainer.__init__).parameters.keys())
trainer_kwargs = dict(
    model=model,
    data_collator=data_collator,
    args=training_args,
    train_dataset=raw_dataset["train"],
    eval_dataset=raw_dataset["test"],
)
if "processing_class" in trainer_sig:
    trainer_kwargs["processing_class"] = processor.feature_extractor
elif "feature_extractor" in trainer_sig:
    trainer_kwargs["feature_extractor"] = processor.feature_extractor

transformers version: 5.13.1
Loading cached raw dataset from disk...
Loading existing vocab...
Loading facebook/wav2vec2-xls-r-300m...


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params: 311,258,269 / 315,468,445 (98.7%)


In [2]:
trainer = transformers.Trainer(**trainer_kwargs)
trainer.train()

Step,Training Loss,Validation Loss
50,13.663098,3.093811
100,11.749909,2.944873


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=100, training_loss=24.898559646606444, metrics={'train_runtime': 446.343, 'train_samples_per_second': 3.585, 'train_steps_per_second': 0.224, 'total_flos': 6.844485757840358e+17, 'train_loss': 24.898559646606444, 'epoch': 7.705882352941177})